# A minor change to avoid data leakgae in PCA
### We shouldnt calculate PCA once using all subjects before K-fold. Instead, calculate PCA separately inside each training fold.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================
# 1. Load dataset
# ============================================================
df = pd.read_csv("wide_diff_all_data.csv")
print("Number of subjects:", len(df))

# ============================================================
# 2. Define diffusion features
# ============================================================
diffusion_substrings = [
    "dki_fa",
    "dki_md",
    "dki_mk",
    "dki_awf"
]

diffusion_cols = [
    col for col in df.columns
    if any(x in col for x in diffusion_substrings)
]

print("Number of diffusion features:", len(diffusion_cols))

# ============================================================
# 3. Prepare outcome and covariates
# ============================================================

# Outcome
y = pd.to_numeric(
    df["DDisc_AUC_40K"],
    errors="coerce"
)


# Covariates
covariates = df[
    [
        "age_group",
        "Gender"
    ]
].copy()


# Convert categorical variables
covariates = pd.get_dummies(
    covariates,
    columns=["age_group", "Gender"],
    drop_first=True
)


# ============================================================
# 4. Create feature matrix
# ============================================================
X_diff = df[diffusion_cols].copy()

# make sure diffusion variables are numeric
X_diff = X_diff.apply(
    pd.to_numeric,
    errors="coerce"
)


# combine MRI + age + sex
X = pd.concat(
    [
        X_diff,
        covariates
    ],
    axis=1
)


# remove subjects with missing values
complete = pd.concat(
    [
        X,
        y
    ],
    axis=1
).dropna()


X = complete.drop(
    columns=["DDisc_AUC_40K"]
)

y = complete["DDisc_AUC_40K"].astype(float)

print("Final subjects:", len(y))
print("Final features:", X.shape[1])

# ============================================================
# 5. Define nested cross validation
# ============================================================

# Outer CV:
# estimates final model performance
outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# Inner CV:
# chooses best hyperparameters
inner_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 6. Create leakage-free pipeline
# ============================================================
pipeline = Pipeline(
    [
        # scaling happens only using training subjects
        (
            "scaler",
            StandardScaler()
        ),

        # PCA is fitted only inside each training fold
        (
            "pca",
            PCA()
        ),

        # prediction model
        (
            "model",
            ElasticNet(
                max_iter=10000
            )
        )
    ]
)


# ============================================================
# 7. Hyperparameter search
# ============================================================

param_grid = {
    # number of PCA components
    "pca__n_components":
        [
            5,
            10,
            15,
            20,
            25
        ],
    # ElasticNet regularization
    "model__alpha":
        np.logspace(-3, 2, 20 ),
    # balance between Ridge and Lasso
    "model__l1_ratio":
        np.linspace(0.1, 0.9, 9 )
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)


# ============================================================
# 8. Nested CV prediction
# ============================================================
y_pred = cross_val_predict( search, X,    y,
    cv=outer_cv,
    n_jobs=-1
)


# ============================================================
# 9. Performance
# ============================================================
rmse = np.sqrt(mean_squared_error(y,y_pred))
mae = mean_absolute_error(y,y_pred)
r2 = r2_score(y,y_pred)


print("==============================")
print("Nested CV Results")
print("==============================")

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
df["DDisc_AUC_40K"].describe()
# ============================================================
# 10. Train final model on all data
#     (optional, after evaluation)
# ============================================================
search.fit(X,y)

print("\nBest parameters:")
print(search.best_params_)

Number of subjects: 691
Number of diffusion features: 96
Final subjects: 691
Final features: 100
Nested CV Results
R²   : -0.0139
RMSE : 0.2855
MAE  : 0.2462

Best parameters:
{'model__alpha': 0.23357214690901212, 'model__l1_ratio': 0.7000000000000001, 'pca__n_components': 5}


## PCA with Ridge Regression

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)


# ============================================================
# 1. Load dataset
# ============================================================
df = pd.read_csv("wide_diff_all_data.csv")
print("Number of subjects:", len(df))

# ============================================================
# 2. Define diffusion features
# ============================================================
diffusion_substrings = [
    "dki_fa",
    "dki_md",
    "dki_mk",
    "dki_awf"
]

diffusion_cols = [
    col for col in df.columns
    if any(x in col for x in diffusion_substrings)
]


print("Number of diffusion features:", len(diffusion_cols))
# ============================================================
# 3. Prepare target
# ============================================================
y = pd.to_numeric(
    df["DDisc_AUC_40K"],
    errors="coerce"
)

# ============================================================
# 4. Prepare covariates
# ============================================================
covariates = df[
    [
        "age_group",
        "Gender"
    ]
].copy()

covariates = pd.get_dummies(
    covariates,
    columns=[
        "age_group",
        "Gender"
    ],
    drop_first=True
)

# ============================================================
# 5. Create full feature matrix
# ============================================================
# Diffusion variables
X_diff = df[diffusion_cols].copy()

X_diff = X_diff.apply(
    pd.to_numeric,
    errors="coerce"
)

# Combine diffusion + covariates
X = pd.concat(
    [
        X_diff,
        covariates
    ],
    axis=1
)


# Remove missing subjects
complete = pd.concat(
    [
        X,
        y
    ],
    axis=1
).dropna()

X = complete.drop(
    columns=[
        "DDisc_AUC_40K"
    ]
)


y = complete["DDisc_AUC_40K"].astype(float)
print("==============================")
print("Final subjects:", len(y))
print("Final avaliable features:", X.shape[1])
print("==============================")

# ============================================================
# 6. Nested Cross Validation
# ============================================================
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# 7. ColumnTransformer
# PCA only on diffusion features
# Age/sex pass directly to Ridge
# ============================================================
preprocessor = ColumnTransformer(
    transformers=[
        (
            "diffusion",
            Pipeline(
                [
                    (
                        "scaler",
                        StandardScaler()
                    ),

                    (
                        "pca",
                        PCA()
                    )
                ]
            ),
            diffusion_cols
        )
    ],
    remainder="passthrough"
)

# ============================================================
# 8. Ridge pipeline
# ============================================================
pipeline = Pipeline(
    [
        (
            "preprocess",
            preprocessor
        ),

        (
            "model",
            Ridge()
        )
    ]
)

# ============================================================
# 9. Hyperparameter search
# ============================================================
param_grid = {
    # Number of PCA components
    "preprocess__diffusion__pca__n_components":
        [
            25,
        ],

    # Ridge regularization
    "model__alpha":
        np.logspace(
            -3,
            4,
            30
        )
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# ============================================================
# 10. Nested CV prediction
# ============================================================
print("Running nested CV...")
y_pred = cross_val_predict(
    search,
    X,
    y,
    cv=outer_cv,
    n_jobs=-1
)

# ============================================================
# 11. Performance
# ============================================================
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
r2 = r2_score( y, y_pred)

print("\n==============================")
print("Nested CV Ridge + PCA Results")
print("==============================")
print(f"R²   : {r2:.4f}")

print(
    f"RMSE : {rmse:.4f}"
)

print(
    f"MAE  : {mae:.4f}"
)


# ============================================================
# 12. Fit final model
# ============================================================
print("\nTraining final model...")
search.fit(X,y)

print("\nBest parameters:")
print(search.best_params_)

Number of subjects: 691
Number of diffusion features: 96
Final subjects: 691
Final features: 100
Running nested CV...

Nested CV Ridge + PCA Results
R²   : -0.0091
RMSE : 0.2848
MAE  : 0.2458

Training final model...

Best parameters:
{'model__alpha': 1082.636733874054, 'preprocess__diffusion__pca__n_components': 25}


## Applying on Test data 

In [5]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error)

# ============================================================
# 1. Load training and unseen test datasets
# ============================================================
train_df = pd.read_csv("wide_diff_all_data.csv")
test_df = pd.read_csv("test_wide_diff_all_data.csv")

print("Training subjects:", len(train_df))
print("Test subjects:", len(test_df))

# ============================================================
# 2. Define diffusion features
# ============================================================
diffusion_substrings = [
    "dki_fa",
    "dki_md",
    "dki_mk",
    "dki_awf"
]

diffusion_cols = [
    col for col in train_df.columns
    if any(x in col for x in diffusion_substrings)
]

print("Number of diffusion features:",len(diffusion_cols))

# ============================================================
# 3. Function to prepare dataset
# ============================================================
def prepare_data(df):
    # Outcome
    y = pd.to_numeric(
        df["DDisc_AUC_40K"],
        errors="coerce"
    )

    # Diffusion features
    X_diff = df[diffusion_cols].copy()
    X_diff = X_diff.apply(
        pd.to_numeric,
        errors="coerce"
    )
    # Age + sex
    cov = df[
        [
            "age_group",
            "Gender"
        ]
    ].copy()

    cov = pd.get_dummies(
        cov,
        columns=[
            "age_group",
            "Gender"
        ],
        drop_first=True
    )

    X = pd.concat(
        [
            X_diff,
            cov
        ],
        axis=1
    )

    return X, y

# Prepare training data
X_train, y_train = prepare_data(train_df)

# Prepare test data
X_test, y_test = prepare_data(test_df)

# ============================================================
# 4. Make sure train and test have identical columns
# ============================================================
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

# Remove missing values
train_complete = pd.concat(
    [
        X_train,
        y_train
    ],
    axis=1
).dropna()


X_train = train_complete.drop(
    columns=[
        "DDisc_AUC_40K"
    ]
)

y_train = train_complete[
    "DDisc_AUC_40K"
].astype(float)

test_complete = pd.concat(
    [
        X_test,
        y_test
    ],
    axis=1
).dropna()


X_test = test_complete.drop(
    columns=[
        "DDisc_AUC_40K"
    ]
)

y_test = test_complete[
    "DDisc_AUC_40K"
].astype(float)

print("==============================")
print("Final datasets")
print("==============================")
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

# ============================================================
# 5. Define preprocessing
#
# PCA only on diffusion features
# ============================================================
preprocessor = ColumnTransformer(
    transformers=[
        (
            "diffusion",
            Pipeline(
                [
                    (
                        "scaler",
                        StandardScaler()
                    ),

                    (
                        "pca",
                        PCA(
                            n_components=25
                        )
                    )
                ]
            ),
            diffusion_cols
        )

    ],
    remainder="passthrough"
)

# ============================================================
# 6. Final Ridge model
# Use parameters selected from CV
# Change these values based on your CV result
# ============================================================
model = Pipeline(
    [
        (
            "preprocess",
            preprocessor
        ),

        (
            "ridge",
            Ridge(
                alpha=10
            )
        )

    ]
)

# ============================================================
# 7. Train ONLY on training subjects
# ============================================================
print("\nTraining final model...")
model.fit( X_train, y_train)

# ============================================================
# 8. Predict unseen test subjects
# ===========================================================
y_pred = model.predict(X_test)

# ============================================================
# 9. Test performance
# ============================================================
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error( y_test, y_pred)

print("\n==============================")
print("Independent Test Results")
print("==============================")
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

# ============================================================
# 10. Save predictions
# ============================================================
results = pd.DataFrame(
    {
        "Subject": test_df["subjectID"],
        "Actual_DD": y_test,
        "Predicted_DD": y_pred
    }
)
results.to_csv(
    "ridge_pca_test_predictions.csv",
    index=False
)

Training subjects: 691
Test subjects: 173
Number of diffusion features: 96
Final datasets
Training: (691, 100)
Testing: (173, 100)

Training final model...

Independent Test Results
R²   : -0.0195
RMSE : 0.2968
MAE  : 0.2565
